In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install camel-tools

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.4
    Uninstalling numpy-2.4.4:
      Successfully uninstalled numpy-2.4.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2

In [1]:
import pandas as pd
import re
from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.disambig.mle import MLEDisambiguator

In [5]:
!pip install --upgrade --force-reinstall numpy

  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
camel-tools 1.5.7 requires numpy<2, but you have numpy 2.4.4 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.


In [2]:
file_path = "/content/predictions.csv"
final_df = pd.read_csv(file_path)
final_df = final_df.reset_index(drop=True)
print(final_df.shape)

(34294, 7)


In [36]:
final_df.head(1)

,Original_Sentence,ref_level_3,ref_level_2,ref_level_1,pred_level_3,pred_level_2,pred_level_1,l3_cwi_details,l3_complex_words,l2_cwi_details,l2_complex_words,l1_cwi_details,l1_complex_words,l3_mlm_substitutions,l2_mlm_substitutions,l1_mlm_substitutions,l3_filtered_substitutions
0,"•• ""عدم الانحياز"" اصطلاح سياسي يعنى الحياد..","""عدم الانحياز"" هو مصطلح سياسي يشير إلى الحياد.","""عدم الانحياز"" هو مصطلح سياسي يعني أن تكون محا...","""عدم الانحياز"" يعني أن تكون محايداً ولا تفضل أ...","""عدم الانحياز"" اصطلاح سياسي يعني الحياد.","""عدم الانحياز"" هو مصطلح سياسي يعني الحياد.","""عدم الانحياز"" هو مصطلح سياسي يعني الحياد.","[{'token': 'عدم', 'lemma': 'عَدَم', 'pos': 'no...","[الانحياز, اصطلاح, سياسي, الحياد]","[{'token': 'عدم', 'lemma': 'عَدَم', 'pos': 'no...","[الانحياز, سياسي, الحياد]","[{'token': 'عدم', 'lemma': 'عَدَم', 'pos': 'no...","[الانحياز, سياسي, الحياد]","[{'original_sentence': '""عدم الانحياز"" اصطلاح ...","[{'original_sentence': '""عدم الانحياز"" هو مصطل...","[{'original_sentence': '""عدم الانحياز"" هو مصطل...","[{'complex_word': 'الانحياز', 'target_pos': 'n..."


In [6]:
list_path = "/content/9203_arabic_words.csv"
list_df = pd.read_csv(list_path)

In [4]:
ARABIC_DIACRITICS = re.compile(r"""
    [\u0617-\u061A]
  | [\u064B-\u0652]
  | [\u0657-\u065F]
  | [\u0670]
  | [\u06D6-\u06ED]
  | [\u0640]
""", re.VERBOSE)

ARABIC_LETTERS_PATTERN = re.compile(r'[\u0600-\u06FF]')
ARABIC_DIGITS_PATTERN = re.compile(r'^[0-9\u0660-\u0669]+$')

def remove_diacritics(text):
    if pd.isna(text):
        return ""
    return re.sub(ARABIC_DIACRITICS, "", str(text))

def normalize_arabic(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = remove_diacritics(text)
    text = re.sub(r"[إأآٱ]", "ا", text)
    text = text.replace("ؤ", "و")
    text = text.replace("ئ", "ي")
    text = text.replace("ى", "ي")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def strip_punct(token):
    """Remove punctuation around token."""
    token = str(token)
    token = re.sub(r"^[^\u0600-\u06FF0-9]+|[^\u0600-\u06FF0-9]+$", "", token)
    return token.strip()

def is_arabic_word(token):
    """Keep only tokens that contain Arabic letters."""
    return bool(ARABIC_LETTERS_PATTERN.search(str(token)))


In [7]:
freq_raw_set = set(
    list_df["Arabic"].dropna().astype(str).str.strip()
)

freq_no_diac_set = set(
    list_df["arabic_no_diacritics"].dropna().astype(str).str.strip()
)

freq_norm_set = set(
    list_df["arabic_normalized"].dropna().astype(str).str.strip()
)


In [8]:
!camel_data -i all

The following packages will be installed: 'morphology-db-egy-r13', 'morphology-db-glf-01', 'disambig-ranking-cache-calima-glf-01', 'disambig-bert-unfactored-egy', 'disambig-mle-calima-egy-r13', 'disambig-ranking-cache-calima-msa-r13', 'sentiment-analysis-mbert', 'disambig-mle-calima-msa-r13', 'disambig-ranking-cache-calima-egy-r13', 'disambig-ranking-cache-calima-lev-01', 'sentiment-analysis-arabert', 'morphology-db-msa-s31', 'disambig-bert-unfactored-msa', 'dialectid-model26', 'disambig-bert-unfactored-glf', 'morphology-db-msa-r13', 'ner-arabert', 'disambig-bert-unfactored-lev', 'dialectid-model6', 'morphology-db-lev-01'
Extracting package 'morphology-db-egy-r13': 100% 67.3M/67.3M [00:00<00:00, 642MB/s]
Extracting package 'morphology-db-glf-01': 100% 7.98M/7.98M [00:00<00:00, 630MB/s]
Extracting package 'disambig-ranking-cache-calima-glf-01': 100% 28.7M/28.7M [00:00<00:00, 518MB/s]
Extracting package 'disambig-bert-unfactored-egy': 100% 446M/446M [00:01<00:00, 414MB/s]
Extracting pack

In [9]:
mle = MLEDisambiguator.pretrained()

def get_best_analysis(token):
    """
    Return best CAMeL analysis dictionary for a token.
    If it fails, return empty dict.
    """
    try:
        disambig = mle.disambiguate([token])
        if not disambig or not disambig[0].analyses:
            return {}
        return disambig[0].analyses[0].analysis
    except:
        return {}

def get_pos(token):
    analysis = get_best_analysis(token)
    return analysis.get("pos", "")

def get_lemma(token):
    """
    Use CAMeL lemma as a fallback lemma signal.
    In your full replication, Farasa can replace/confirm this later.
    """
    analysis = get_best_analysis(token)
    lemma = analysis.get("lex", "") or analysis.get("lemma", "")
    return str(lemma).strip()

In [10]:
# The paper excludes prepositions before checking complexity.
# Since automatic POS tagging is not always perfect for function words,
# we also keep a manual Arabic preposition list as a safety layer.

PREPOSITIONS = {
    "في", "من", "الى", "إلى", "عن", "على", "ب", "ل", "ك", "حتى",
    "منذ", "مذ", "خلال", "دون", "عند", "بين", "نحو", "حول", "مع",
    "لدى", "تحت", "فوق", "وراء", "أمام", "داخل", "خارج", "بعد", "قبل"
}

# We also normalize them once so comparison is consistent
PREPOSITIONS_NORM = {normalize_arabic(p) for p in PREPOSITIONS}

In [11]:
def is_number_token(token, pos_tag=""):
    token = str(token).strip()
    pos_tag = str(pos_tag).lower()
    return ARABIC_DIGITS_PATTERN.match(token) is not None or pos_tag.startswith("num")

def is_proper_noun(pos_tag=""):
    pos_tag = str(pos_tag).lower()
    return pos_tag.startswith("noun_prop")

def is_preposition(token, pos_tag=""):
    pos_tag = str(pos_tag).lower()
    if normalize_arabic(token) in PREPOSITIONS_NORM:
        return True
    if pos_tag.startswith("prep"):
        return True
    return False

In [12]:
import re
import pandas as pd

def split_variants_enhanced(text):
    """
    Splits a cell into individual words to ensure every single word
    is captured in the lookup set.
    """
    if pd.isna(text):
        return []

    text = str(text).strip()
    if not text:
        return []

    # 1. Split by common separators (Arabic comma, English comma, semicolon, slash)
    parts = re.split(r'[،,;/]+', text)

    final_words = []
    for part in parts:
        # 2. Split each part by spaces to extract individual words
        words = part.split()
        final_words.extend(words)

    # 3. Clean words from punctuation (like parentheses or periods)
    clean_words = [re.sub(r'[^\w\s]', '', w).strip() for w in final_words if w.strip()]

    return clean_words

def build_variant_set(series):
    """
    Iterates through a column and builds a unique set of all
    individual words found in those cells.
    """
    values = set()
    for cell in series.dropna():
        # Get the list of individual words from the enhanced splitter
        for item in split_variants_enhanced(cell):
            values.add(item)
    return values


freq_raw_set = build_variant_set(list_df["Arabic"])

# Example check:
# Even if the Excel cell says "سباق السيارات", this will now return True
print("السيارات" in freq_raw_set)

True


In [13]:
def exists_in_frequency_list(token, lemma=""):
    """
    A word is considered found if ANY of these match the list:
    - raw token
    - token without diacritics
    - normalized token
    - raw lemma
    - lemma without diacritics
    - normalized lemma
    """
    candidates = set()

    token = strip_punct(token)
    lemma = strip_punct(lemma)

    if token:
        candidates.add(str(token).strip())
        candidates.add(remove_diacritics(token).strip())
        candidates.add(normalize_arabic(token).strip())

    if lemma:
        candidates.add(str(lemma).strip())
        candidates.add(remove_diacritics(lemma).strip())
        candidates.add(normalize_arabic(lemma).strip())

    # remove empty items
    candidates = {c for c in candidates if c}

    for c in candidates:
        if c in freq_raw_set or c in freq_no_diac_set or c in freq_norm_set:
            return True, sorted(candidates)

    return False, sorted(candidates)

In [14]:
def identify_complex_words(sentence):
    """
    For one sentence:
    - tokenize
    - clean punctuation
    - exclude proper nouns, prepositions, and numbers
    - check token and lemma against frequency list
    - if found in the frequency list -> exclude from complexity candidates
    - if not found -> treat as complex
    """
    if pd.isna(sentence):
        return []

    tokens = simple_word_tokenize(str(sentence))
    results = []

    for token in tokens:
        cleaned_token = strip_punct(token)

        # Skip empty or punctuation-only tokens
        if not cleaned_token:
            continue

        # Skip non-Arabic tokens
        if not is_arabic_word(cleaned_token):
            continue

        analysis = get_best_analysis(cleaned_token)
        pos_tag = analysis.get("pos", "")
        lemma = analysis.get("lex", "") or analysis.get("lemma", "")

        # Exclude proper nouns
        if is_proper_noun(pos_tag):
            results.append({
                "token": cleaned_token,
                "lemma": lemma,
                "pos": pos_tag,
                "excluded": True,
                "exclude_reason": "proper_noun",
                "lookup_candidates": [],
                "in_frequency_list": None,
                "is_complex": False
            })
            continue

        # Exclude prepositions
        if is_preposition(cleaned_token, pos_tag):
            results.append({
                "token": cleaned_token,
                "lemma": lemma,
                "pos": pos_tag,
                "excluded": True,
                "exclude_reason": "preposition",
                "lookup_candidates": [],
                "in_frequency_list": None,
                "is_complex": False
            })
            continue

        # Exclude numbers
        if is_number_token(cleaned_token, pos_tag):
            results.append({
                "token": cleaned_token,
                "lemma": lemma,
                "pos": pos_tag,
                "excluded": True,
                "exclude_reason": "number",
                "lookup_candidates": [],
                "in_frequency_list": None,
                "is_complex": False
            })
            continue

        # Check whether the word exists in the frequency list
        in_freq, lookup_candidates = exists_in_frequency_list(cleaned_token, lemma)

        # If found in the list, exclude it from complex-word candidates
        if in_freq:
            results.append({
                "token": cleaned_token,
                "lemma": lemma,
                "pos": pos_tag,
                "excluded": True,
                "exclude_reason": "in_frequency_list",
                "lookup_candidates": lookup_candidates,
                "in_frequency_list": True,
                "is_complex": False
            })
            continue

        # Otherwise, the word is treated as complex
        results.append({
            "token": cleaned_token,
            "lemma": lemma,
            "pos": pos_tag,
            "excluded": False,
            "exclude_reason": "",
            "lookup_candidates": lookup_candidates,
            "in_frequency_list": False,
            "is_complex": True
        })

    return results

In [15]:
# This creates:
# - cwi_details: full token-level analysis
# - complex_words: only the tokens marked as complex
final_df["l3_cwi_details"] = final_df["pred_level_3"].apply(identify_complex_words)
final_df["l3_complex_words"] = final_df["l3_cwi_details"].apply(
    lambda rows: [r["token"] for r in rows if r["is_complex"]]
)

final_df["l2_cwi_details"] = final_df["pred_level_2"].apply(identify_complex_words)
final_df["l2_complex_words"] = final_df["l2_cwi_details"].apply(
    lambda rows: [r["token"] for r in rows if r["is_complex"]]
)

final_df["l1_cwi_details"] = final_df["pred_level_1"].apply(identify_complex_words)
final_df["l1_complex_words"] = final_df["l1_cwi_details"].apply(
    lambda rows: [r["token"] for r in rows if r["is_complex"]]
)

In [16]:
# quick preview for all hybrid levels
for i in range(min(5, len(final_df))):
    print("=" * 100)
    print("Original_Text:", final_df.loc[i, "Original_Sentence"])

    for level in ["3", "2", "1"]:
        print("-" * 80)
        print(f"AraBART Pred L{level}:", final_df.loc[i, f"pred_level_{level}"])
        print(f"Complex words L{level}:", final_df.loc[i, f"l{level}_complex_words"])
        print(f"Details L{level}:")
        for row in final_df.loc[i, f"l{level}_cwi_details"]:
            print(row)

Original_Text: •• "عدم الانحياز" اصطلاح سياسي يعنى الحياد..
--------------------------------------------------------------------------------
AraBART Pred L3: "عدم الانحياز" اصطلاح سياسي يعني الحياد.
Complex words L3: ['الانحياز', 'اصطلاح', 'سياسي', 'الحياد']
Details L3:
{'token': 'عدم', 'lemma': 'عَدَم', 'pos': 'noun', 'excluded': True, 'exclude_reason': 'in_frequency_list', 'lookup_candidates': ['عدم', 'عَدَم'], 'in_frequency_list': True, 'is_complex': False}
{'token': 'الانحياز', 'lemma': 'ٱِنْحِياز', 'pos': 'noun', 'excluded': False, 'exclude_reason': '', 'lookup_candidates': ['الانحياز', 'انحياز', 'ٱنحياز', 'ٱِنْحِياز'], 'in_frequency_list': False, 'is_complex': True}
{'token': 'اصطلاح', 'lemma': 'ٱِصْطِلاح', 'pos': 'noun', 'excluded': False, 'exclude_reason': '', 'lookup_candidates': ['اصطلاح', 'ٱصطلاح', 'ٱِصْطِلاح'], 'in_frequency_list': False, 'is_complex': True}
{'token': 'سياسي', 'lemma': 'سِياسِيّ', 'pos': 'adj', 'excluded': False, 'exclude_reason': '', 'lookup_candidates': [

In [17]:
# Stage 2: Generate substitution candidates with Arabic-BERT MLM

In [18]:
import re
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForMaskedLM

In [19]:
# 1) Load Arabic-BERT masked language model
# ------------------------------------------------------------
# Practical Arabic-BERT choice from the asafaya family.
# This is used here as a masked language model to generate
# substitution candidates for a masked complex word.

MODEL_NAME = "asafaya/bert-base-arabic"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Loaded model on:", device)
print("Mask token:", tokenizer.mask_token)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Some weights of the model checkpoint at asafaya/bert-base-arabic were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Loaded model on: cuda
Mask token: [MASK]


In [20]:
def mask_first_occurrence(sentence, target_word, mask_token):
    """
    Replace only the first exact occurrence of target_word in sentence
    with [MASK]. This avoids replacing every occurrence.
    """
    if not sentence or not target_word:
        return sentence, False

    # Escape target for regex safety
    pattern = re.escape(target_word)

    # Replace only first match
    masked_sentence, n = re.subn(pattern, mask_token, sentence, count=1)
    return masked_sentence, (n > 0)

In [21]:
def is_valid_candidate(token):
    """
    Keep only clean substitution candidates.
    We filter out:
    - special tokens
    - subword pieces like ##...
    - empty tokens
    - punctuation-only tokens
    - obvious non-Arabic fragments
    """
    if token is None:
        return False

    token = str(token).strip()

    if not token:
        return False

    # Skip special tokens and wordpieces
    if token in tokenizer.all_special_tokens:
        return False
    if token.startswith("##"):
        return False

    # Skip punctuation-only
    if re.fullmatch(r"[^\w\u0600-\u06FF]+", token):
        return False

    # Keep tokens that contain at least one Arabic character
    if not re.search(r"[\u0600-\u06FF]", token):
        return False

    return True


In [22]:
def generate_top5_substitutions(sentence, complex_word, top_k=5, search_k=50):
    """
    For one sentence + one complex word:
    - replace the word with [MASK]
    - run Arabic-BERT MLM
    - return top 5 candidate substitutions with probabilities

    Notes:
    - This works best when the complex word is represented as one mask.
    - If the target word is not found in the sentence, returns empty list.
    """
    masked_sentence, replaced = mask_first_occurrence(
        sentence=sentence,
        target_word=complex_word,
        mask_token=tokenizer.mask_token
    )

    if not replaced:
        return {
            "original_sentence": sentence,
            "complex_word": complex_word,
            "masked_sentence": sentence,
            "found_and_masked": False,
            "substitutions": []
        }

    # Tokenize
    inputs = tokenizer(masked_sentence, return_tensors="pt").to(device)

    # Find [MASK] position
    mask_positions = (inputs["input_ids"][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]

    if len(mask_positions) == 0:
        return {
            "original_sentence": sentence,
            "complex_word": complex_word,
            "masked_sentence": masked_sentence,
            "found_and_masked": False,
            "substitutions": []
        }

    mask_index = mask_positions[0].item()

    # Run model
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, mask_index]

    # Convert logits to probabilities
    probs = torch.softmax(logits, dim=-1)

    # Take a larger pool first, then filter to valid Arabic candidates
    top_ids = torch.topk(probs, k=search_k).indices.tolist()

    candidates = []
    seen = set()

    for token_id in top_ids:
        token = tokenizer.convert_ids_to_tokens(token_id)
        prob = probs[token_id].item()

        if not is_valid_candidate(token):
            continue

        # Avoid duplicates
        if token in seen:
            continue

        seen.add(token)

        candidates.append({
            "candidate": token,
            "probability": prob
        })

        if len(candidates) == top_k:
            break

    return {
        "original_sentence": sentence,
        "complex_word": complex_word,
        "masked_sentence": masked_sentence,
        "found_and_masked": True,
        "substitutions": candidates
    }

In [23]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Compute Capability: {torch.cuda.get_device_capability(0)}")

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: Tesla T4
GPU Compute Capability: (7, 5)


In [24]:
from tqdm.auto import tqdm

def generate_mlm_for_level(df, sentence_col, complex_col, output_col):
    all_results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {sentence_col}"):
        sentence = row[sentence_col]
        complex_words = row[complex_col]

        if not isinstance(complex_words, list) or len(complex_words) == 0:
            all_results.append([])
            continue

        row_results = []
        for cw in complex_words:
            result = generate_top5_substitutions(sentence, cw, top_k=5, search_k=50)
            row_results.append(result)

        all_results.append(row_results)

    df[output_col] = all_results
    return df

In [25]:
final_df = generate_mlm_for_level(
    final_df,
    sentence_col="pred_level_3",
    complex_col="l3_complex_words",
    output_col="l3_mlm_substitutions"
)

final_df = generate_mlm_for_level(
    final_df,
    sentence_col="pred_level_2",
    complex_col="l2_complex_words",
    output_col="l2_mlm_substitutions"
)

final_df = generate_mlm_for_level(
    final_df,
    sentence_col="pred_level_1",
    complex_col="l1_complex_words",
    output_col="l1_mlm_substitutions"
)

Processing pred_level_3:   0%|          | 0/34294 [00:00<?, ?it/s]

Processing pred_level_2:   0%|          | 0/34294 [00:00<?, ?it/s]

Processing pred_level_1:   0%|          | 0/34294 [00:00<?, ?it/s]

In [26]:
# Preview a few rows for all levels
for i in range(min(3, len(final_df))):
    print("=" * 100)
    print("Original_Text:", final_df.loc[i, "Original_Sentence"])

    for level in ["3", "2", "1"]:
        print("\n" + "-" * 80)
        print(f"LEVEL {level}")
        print(f"AraBART prediction:", final_df.loc[i, f"pred_level_{level}"])
        print(f"Complex words:", final_df.loc[i, f"l{level}_complex_words"])
        print("MLM substitutions:")

        for item in final_df.loc[i, f"l{level}_mlm_substitutions"]:
            print("-" * 60)
            print("Complex word:", item["complex_word"])
            print("Masked sentence:", item["masked_sentence"])
            print("Candidates:")
            for cand in item["substitutions"]:
                print(cand)

Original_Text: •• "عدم الانحياز" اصطلاح سياسي يعنى الحياد..

--------------------------------------------------------------------------------
LEVEL 3
AraBART prediction: "عدم الانحياز" اصطلاح سياسي يعني الحياد.
Complex words: ['الانحياز', 'اصطلاح', 'سياسي', 'الحياد']
MLM substitutions:
------------------------------------------------------------
Complex word: الانحياز
Masked sentence: "عدم [MASK]" اصطلاح سياسي يعني الحياد.
Candidates:
{'candidate': 'الرضا', 'probability': 0.06793669611215591}
{'candidate': 'التدخل', 'probability': 0.045033395290374756}
{'candidate': 'الاحترام', 'probability': 0.03276299312710762}
{'candidate': 'التمييز', 'probability': 0.028916973620653152}
{'candidate': 'الاستقرار', 'probability': 0.02701532281935215}
------------------------------------------------------------
Complex word: اصطلاح
Masked sentence: "عدم الانحياز" [MASK] سياسي يعني الحياد.
Candidates:
{'candidate': 'مصطلح', 'probability': 0.19418413937091827}
{'candidate': 'موقف', 'probability': 0.0803

In [27]:
model.save_pretrained("/kaggle/working/substitution_model")
tokenizer.save_pretrained("/kaggle/working/substitution_model")

('/kaggle/working/substitution_model/tokenizer_config.json',
 '/kaggle/working/substitution_model/special_tokens_map.json',
 '/kaggle/working/substitution_model/vocab.txt',
 '/kaggle/working/substitution_model/added_tokens.json',
 '/kaggle/working/substitution_model/tokenizer.json')

In [28]:
import shutil
shutil.make_archive("/kaggle/working/substitution_model", 'zip', "/kaggle/working/substitution_model")

'/kaggle/working/substitution_model.zip'

In [29]:
# Stage 3: Rule-based filtering + final candidate selection
# Based on the paper:
# 1) candidate lemma must be different from target lemma
# 2) candidate POS must be the same as target POS
# 3) candidate must exist in the frequency list
# 4) choose highest-probability surviving candidate

In [30]:
# safely get POS and lemma for a word
def get_token_pos_and_lemma(token):
    """
    Return (pos, lemma) using the same analysis function you already use.
    """
    analysis = get_best_analysis(token)
    pos_tag = analysis.get("pos", "")
    lemma = analysis.get("lex", "") or analysis.get("lemma", "")
    return pos_tag, lemma


In [31]:
def normalize_for_comparison(text):
    """
    Light normalization for comparing target lemma vs candidate lemma.
    """
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = remove_diacritics(text)
    text = normalize_arabic(text)
    return text


In [32]:
def filter_and_select_substitution(one_result):
    """
    Input:
      one_result = one item from mlm_substitutions
      Example:
      {
        'original_sentence': ...,
        'complex_word': ...,
        'masked_sentence': ...,
        'found_and_masked': True,
        'substitutions': [{'candidate': ..., 'probability': ...}, ...]
      }

    Output:
      dictionary containing:
      - target word info
      - all candidate analyses
      - valid candidates only
      - selected best substitution
    """
    complex_word = one_result.get("complex_word", "")
    substitutions = one_result.get("substitutions", [])


    # Analyze target word
    target_pos, target_lemma = get_token_pos_and_lemma(complex_word)
    target_lemma_norm = normalize_for_comparison(target_lemma)
    target_word_norm = normalize_for_comparison(complex_word)

    all_candidates = []
    valid_candidates = []


    for cand in substitutions:
        candidate_word = cand.get("candidate", "")
        probability = cand.get("probability", 0.0)

        cand_pos, cand_lemma = get_token_pos_and_lemma(candidate_word)
        cand_lemma_norm = normalize_for_comparison(cand_lemma)
        cand_word_norm = normalize_for_comparison(candidate_word)

        # Rule 1: lemma must be different from target lemma
        lemma_is_different = (
            cand_lemma_norm != target_lemma_norm
            and cand_word_norm != target_word_norm
        )
        # Rule 2: POS must match target POS
        same_pos = (cand_pos == target_pos)

        # Rule 3: candidate must exist in frequency list
        in_freq, lookup_candidates = exists_in_frequency_list(candidate_word, cand_lemma)

        candidate_info = {
            "candidate": candidate_word,
            "probability": probability,
            "candidate_pos": cand_pos,
            "candidate_lemma": cand_lemma,
            "candidate_lookup_candidates": lookup_candidates,
            "lemma_is_different": lemma_is_different,
            "same_pos": same_pos,
            "in_frequency_list": in_freq,
            "is_valid": lemma_is_different and same_pos and in_freq
        }
        all_candidates.append(candidate_info)

        if candidate_info["is_valid"]:
            valid_candidates.append(candidate_info)

    # Select highest-probability valid candidate
    selected = None
    if valid_candidates:
        selected = max(valid_candidates, key=lambda x: x["probability"])

    return {
        "complex_word": complex_word,
        "target_pos": target_pos,
        "target_lemma": target_lemma,
        "all_candidates": all_candidates,
        "valid_candidates": valid_candidates,
        "selected_substitution": selected
    }

In [37]:
def filter_substitutions_list(mlm_results):
    if not isinstance(mlm_results, list) or len(mlm_results) == 0:
        return []

    filtered_results = []
    for one_result in mlm_results:
        filtered_results.append(filter_and_select_substitution(one_result))

    return filtered_results


final_df["l3_filtered_substitutions"] = final_df["l3_mlm_substitutions"].apply(filter_substitutions_list)
final_df["l2_filtered_substitutions"] = final_df["l2_mlm_substitutions"].apply(filter_substitutions_list)
final_df["l1_filtered_substitutions"] = final_df["l1_mlm_substitutions"].apply(filter_substitutions_list)

In [39]:
# Extract only final chosen substitutions
def extract_final_substitutions(filtered_results):
    """
    Return a compact list of final chosen substitutions per row.
    """
    if not isinstance(filtered_results, list):
        return []

    finals = []
    for item in filtered_results:
        selected = item.get("selected_substitution", None)
        if selected is not None:
            finals.append({
                "complex_word": item["complex_word"],
                "replacement": selected["candidate"],
                "probability": selected["probability"],
                "target_pos": item["target_pos"],
                "target_lemma": item["target_lemma"],
                "candidate_pos": selected["candidate_pos"],
                "candidate_lemma": selected["candidate_lemma"]
            })
    return finals


final_df["l3_final_substitutions"] = final_df["l3_filtered_substitutions"].apply(extract_final_substitutions)
final_df["l2_final_substitutions"] = final_df["l2_filtered_substitutions"].apply(extract_final_substitutions)
final_df["l1_final_substitutions"] = final_df["l1_filtered_substitutions"].apply(extract_final_substitutions)

In [40]:
# Preview final substitutions for all hybrid levels
for i in range(min(3, len(final_df))):
    print("=" * 100)
    print("Original:", final_df.loc[i, "Original_Sentence"])

    for level in ["3", "2", "1"]:
        print("\n" + "-" * 80)
        print(f"LEVEL {level}")
        print(f"AraBART Pred L{level}:", final_df.loc[i, f"pred_level_{level}"])
        print(f"Complex words L{level}:", final_df.loc[i, f"l{level}_complex_words"])

        print("\nFinal substitutions:")
        for item in final_df.loc[i, f"l{level}_final_substitutions"]:
            print(item)

        print("\nFiltered details:")
        for item in final_df.loc[i, f"l{level}_filtered_substitutions"]:
            print("-" * 60)
            print("Complex word:", item["complex_word"])
            print("Target POS:", item["target_pos"])
            print("Target lemma:", item["target_lemma"])
            print("Selected substitution:", item["selected_substitution"])

Original: •• "عدم الانحياز" اصطلاح سياسي يعنى الحياد..

--------------------------------------------------------------------------------
LEVEL 3
AraBART Pred L3: "عدم الانحياز" اصطلاح سياسي يعني الحياد.
Complex words L3: ['الانحياز', 'اصطلاح', 'سياسي', 'الحياد']

Final substitutions:
{'complex_word': 'الانحياز', 'replacement': 'الرضا', 'probability': 0.06793669611215591, 'target_pos': 'noun', 'target_lemma': 'ٱِنْحِياز', 'candidate_pos': 'noun', 'candidate_lemma': 'رِضَى'}
{'complex_word': 'اصطلاح', 'replacement': 'مصطلح', 'probability': 0.19418413937091827, 'target_pos': 'noun', 'target_lemma': 'ٱِصْطِلاح', 'candidate_pos': 'noun', 'candidate_lemma': 'مُصْطَلَح'}
{'complex_word': 'الحياد', 'replacement': 'الاستقلال', 'probability': 0.0541318915784359, 'target_pos': 'noun', 'target_lemma': 'حِياد', 'candidate_pos': 'noun', 'candidate_lemma': 'ٱِسْتِقْلال'}

Filtered details:
------------------------------------------------------------
Complex word: الانحياز
Target POS: noun
Target lemm

In [41]:
import re
import pandas as pd


In [42]:
# Safe single replacement
def replace_first_exact_occurrence(sentence, old_word, new_word):
    """
    Replace only the first exact occurrence of old_word in sentence.
    Returns:
      new_sentence, replaced_flag
    """
    if pd.isna(sentence):
        return sentence, False

    sentence = str(sentence)
    old_word = str(old_word)
    new_word = str(new_word)

    if not old_word or not new_word:
        return sentence, False

    pattern = re.escape(old_word)
    new_sentence, n = re.subn(pattern, new_word, sentence, count=1)

    return new_sentence, (n > 0)

In [43]:
def build_hybrid_sentence_from_pred(row, pred_col, subs_col):
    """
    For one row:
    - start from AraBART prediction column
    - apply lexical substitutions
    - return hybrid simplified sentence
    """
    base_sentence = row[pred_col]
    substitutions = row[subs_col]

    if pd.isna(base_sentence):
        return ""

    sentence = str(base_sentence)

    if not isinstance(substitutions, list) or len(substitutions) == 0:
        return sentence

    seen_pairs = set()
    ordered_subs = []

    for item in substitutions:
        complex_word = item.get("complex_word", "")
        replacement = item.get("replacement", "")
        key = (complex_word, replacement)

        if key not in seen_pairs:
            seen_pairs.add(key)
            ordered_subs.append(item)

    for item in ordered_subs:
        complex_word = item.get("complex_word", "")
        replacement = item.get("replacement", "")

        if complex_word and replacement:
            sentence, _ = replace_first_exact_occurrence(
                sentence=sentence,
                old_word=complex_word,
                new_word=replacement
            )

    return sentence

In [44]:
#Apply to dataframe
final_df["hybrid_level_3"] = final_df.apply(
    lambda row: build_hybrid_sentence_from_pred(row, "pred_level_3", "l3_final_substitutions"),
    axis=1
)

final_df["hybrid_level_2"] = final_df.apply(
    lambda row: build_hybrid_sentence_from_pred(row, "pred_level_2", "l2_final_substitutions"),
    axis=1
)

final_df["hybrid_level_1"] = final_df.apply(
    lambda row: build_hybrid_sentence_from_pred(row, "pred_level_1", "l1_final_substitutions"),
    axis=1
)

In [45]:
#store a compact replacement map for inspection
def build_replacement_map(substitutions):
    """
    Create a simple readable list of replacements for debugging.
    """
    if not isinstance(substitutions, list):
        return []

    return [
        f"{item.get('complex_word', '')} -> {item.get('replacement', '')}"
        for item in substitutions
    ]

final_df["l3_replacement_map"] = final_df["l3_final_substitutions"].apply(build_replacement_map)
final_df["l2_replacement_map"] = final_df["l2_final_substitutions"].apply(build_replacement_map)
final_df["l1_replacement_map"] = final_df["l1_final_substitutions"].apply(build_replacement_map)

In [46]:
for i in range(min(10, len(final_df))):
    print(f"\n{'='*80}")
    print("ORIGINAL       :", final_df.loc[i, "Original_Sentence"])
    print("PRED LEVEL 3   :", final_df.loc[i, "pred_level_3"])
    print("HYBRID LEVEL 3 :", final_df.loc[i, "hybrid_level_3"])
    print("REF LEVEL 3    :", final_df.loc[i, "ref_level_3"])
    print("L3 MAP         :", final_df.loc[i, "l3_replacement_map"])

    print("-"*80)
    print("PRED LEVEL 2   :", final_df.loc[i, "pred_level_2"])
    print("HYBRID LEVEL 2 :", final_df.loc[i, "hybrid_level_2"])
    print("REF LEVEL 2    :", final_df.loc[i, "ref_level_2"])
    print("L2 MAP         :", final_df.loc[i, "l2_replacement_map"])

    print("-"*80)
    print("PRED LEVEL 1   :", final_df.loc[i, "pred_level_1"])
    print("HYBRID LEVEL 1 :", final_df.loc[i, "hybrid_level_1"])
    print("REF LEVEL 1    :", final_df.loc[i, "ref_level_1"])
    print("L1 MAP         :", final_df.loc[i, "l1_replacement_map"])


ORIGINAL       : •• "عدم الانحياز" اصطلاح سياسي يعنى الحياد..
PRED LEVEL 3   : "عدم الانحياز" اصطلاح سياسي يعني الحياد.
HYBRID LEVEL 3 : "عدم الرضا" مصطلح سياسي يعني الاستقلال.
REF LEVEL 3    : "عدم الانحياز" هو مصطلح سياسي يشير إلى الحياد.
L3 MAP         : ['الانحياز -> الرضا', 'اصطلاح -> مصطلح', 'الحياد -> الاستقلال']
--------------------------------------------------------------------------------
PRED LEVEL 2   : "عدم الانحياز" هو مصطلح سياسي يعني الحياد.
HYBRID LEVEL 2 : "عدم الرضا" هو مصطلح عربي يعني الاستقلال.
REF LEVEL 2    : "عدم الانحياز" هو مصطلح سياسي يعني أن تكون محايدًا.
L2 MAP         : ['الانحياز -> الرضا', 'سياسي -> عربي', 'الحياد -> الاستقلال']
--------------------------------------------------------------------------------
PRED LEVEL 1   : "عدم الانحياز" هو مصطلح سياسي يعني الحياد.
HYBRID LEVEL 1 : "عدم الرضا" هو مصطلح عربي يعني الاستقلال.
REF LEVEL 1    : "عدم الانحياز" يعني أن تكون محايداً ولا تفضل أحداً.
L1 MAP         : ['الانحياز -> الرضا', 'سياسي -> عربي', 'الح

In [47]:
final_df.to_csv("/kaggle/working/final_hybrid_df_with_simplifications.csv", index=False)
print("Saved to /kaggle/working/final_hybrid_df_with_simplifications.csv")

Saved to /kaggle/working/final_hybrid_df_with_simplifications.csv


In [48]:
from google.colab import files

files.download("/kaggle/working/final_hybrid_df_with_simplifications.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>